# Melbourne Airbnb Price Prediction using Regression Models

This notebook presents a cleaned, portfolio-friendly machine learning workflow for predicting nightly Airbnb listing prices in Melbourne using listing, host, location, room, and review-related features.

The original project was completed as part of Computational Machine Learning coursework. The assessment dataset is **not included** in this repository because it was provided only for assessment use. This notebook keeps the workflow and modelling structure while avoiding redistribution of restricted data.

## Project Goal

The goal is to predict the nightly price of an Airbnb listing based on structured features such as location, room type, accommodation capacity, bedrooms, bathrooms, reviews, booking options, and host/listing information.

This is a **supervised regression** task because the target variable, `price`, is numerical.

## Data Availability

To run this notebook locally, place compatible data files in a `data/` folder:

- `data/train_data.csv` — training data containing the target column `price`
- `data/test_data.csv` — optional unseen data without `price`

The dataset used for the original assignment is not uploaded here due to data-use restrictions.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [ ]:
RANDOM_STATE = 42
DATA_DIR = Path("data")
TRAIN_PATH = DATA_DIR / "train_data.csv"
TEST_PATH = DATA_DIR / "test_data.csv"
TARGET = "price"

In [ ]:
def load_data(train_path=TRAIN_PATH, test_path=TEST_PATH):
    """Load project data if available locally."""
    if not train_path.exists():
        raise FileNotFoundError(
            f"Could not find {train_path}. "
            "The original assessment dataset is not included in this repository. "
            "Place a compatible train_data.csv file in the data/ folder to run the notebook."
        )

    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path) if test_path.exists() else None
    return train_df, test_df

# Uncomment to load local data when available.
# train_df, test_df = load_data()

## Data Inspection

The original dataset contained both numerical and categorical variables. The target variable was `price`. The workflow below avoids displaying raw rows so that the notebook remains suitable for a public portfolio repository.

In [ ]:
def inspect_data(df, target=TARGET):
    """Return a compact, non-sensitive summary of the dataset."""
    summary = {
        "rows": df.shape[0],
        "columns": df.shape[1],
        "missing_values_total": int(df.isna().sum().sum()),
        "target_present": target in df.columns,
    }
    return pd.Series(summary)

# Example after loading data:
# inspect_data(train_df)

In [ ]:
def get_feature_groups(df, target=TARGET):
    """Identify numerical and categorical features for preprocessing."""
    X = df.drop(columns=[target]) if target in df.columns else df.copy()
    numerical_features = X.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()
    categorical_features = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
    return numerical_features, categorical_features

# numerical_features, categorical_features = get_feature_groups(train_df)
# numerical_features, categorical_features

## Exploratory Data Analysis

The original analysis showed three important patterns:

- Listing prices were right-skewed, with many lower-priced listings and a smaller number of expensive listings.
- Room type was strongly related to price; entire homes/apartments were more expensive on average than private or shared rooms.
- Capacity and size-related features such as bedrooms, accommodates, beds, and bathrooms had the strongest numerical correlations with price.

The functions below reproduce those checks when compatible data is available locally.

In [ ]:
def plot_price_distribution(df, target=TARGET):
    plt.figure(figsize=(8, 5))
    plt.hist(df[target], bins=50)
    plt.title("Distribution of Listing Price")
    plt.xlabel("Price")
    plt.ylabel("Frequency")
    plt.show()

# plot_price_distribution(train_df)

In [ ]:
def plot_average_price_by_room_type(df, target=TARGET):
    room_price = df.groupby("room_type")[target].mean().sort_values()

    plt.figure(figsize=(8, 5))
    room_price.plot(kind="bar")
    plt.title("Average Price by Room Type")
    plt.xlabel("Room Type")
    plt.ylabel("Average Price")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

    return room_price

# plot_average_price_by_room_type(train_df)

In [ ]:
def numeric_correlations_with_price(df, target=TARGET, top_n=10):
    numeric_df = df.select_dtypes(include=["int64", "float64", "int32", "float32"])
    correlations = numeric_df.corr(numeric_only=True)[target].drop(target).sort_values(key=abs, ascending=False)
    return correlations.head(top_n)

# numeric_correlations_with_price(train_df)

## Preprocessing and Evaluation Design

The training data is split into training and validation sets. The separate test file, if available, should only be used after model selection.

Preprocessing steps:

- Numerical features are standardised using `StandardScaler`.
- Categorical features are encoded using `OneHotEncoder`.
- Unknown categories are ignored to make the pipeline robust to unseen category values.

In [ ]:
def make_one_hot_encoder():
    """Create a OneHotEncoder compatible with different scikit-learn versions."""
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def build_preprocessor(numerical_features, categorical_features):
    return ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), numerical_features),
            ("cat", make_one_hot_encoder(), categorical_features),
        ]
    )

In [ ]:
def split_features_target(df, target=TARGET):
    X = df.drop(columns=[target])
    y = df[target]
    return X, y


def make_train_validation_split(X, y, test_size=0.2, random_state=RANDOM_STATE):
    return train_test_split(X, y, test_size=test_size, random_state=random_state)

## Models

Three regression models are compared:

1. Linear Regression — baseline model
2. Ridge Regression — L2-regularised linear model
3. Lasso Regression — L1-regularised linear model

The models are evaluated using:

- **RMSE** — penalises larger errors more strongly
- **MAE** — average absolute dollar error
- **R²** — proportion of target variation explained by the model

In [ ]:
def evaluate_regression_model(model, X_train, y_train, X_val, y_val):
    model.fit(X_train, y_train)
    predictions = model.predict(X_val)

    rmse = np.sqrt(mean_squared_error(y_val, predictions))
    mae = mean_absolute_error(y_val, predictions)
    r2 = r2_score(y_val, predictions)

    return {
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2,
    }

In [ ]:
def compare_models(train_df, target=TARGET):
    numerical_features, categorical_features = get_feature_groups(train_df, target=target)
    preprocessor = build_preprocessor(numerical_features, categorical_features)

    X, y = split_features_target(train_df, target=target)
    X_train, X_val, y_train, y_val = make_train_validation_split(X, y)

    models = {
        "Linear Regression": LinearRegression(),
        "Ridge Regression (alpha=1)": Ridge(alpha=1),
        "Lasso Regression (alpha=1)": Lasso(alpha=1, max_iter=10000),
    }

    results = []
    fitted_models = {}

    for name, estimator in models.items():
        pipeline = Pipeline(steps=[
            ("preprocessor", preprocessor),
            ("model", estimator),
        ])
        metrics = evaluate_regression_model(pipeline, X_train, y_train, X_val, y_val)
        results.append({"Model": name, **metrics})
        fitted_models[name] = pipeline

    results_df = pd.DataFrame(results).sort_values("RMSE")
    return results_df, fitted_models

# results_df, fitted_models = compare_models(train_df)
# results_df

## Ridge Hyperparameter Tuning

Ridge Regression was tuned by testing multiple values of `alpha`, the regularisation strength. Small values behave similarly to ordinary Linear Regression, while very large values may oversimplify the model.

In [ ]:
def tune_ridge_alpha(train_df, alpha_values=None, target=TARGET):
    if alpha_values is None:
        alpha_values = [0.01, 0.1, 1, 10, 100]

    numerical_features, categorical_features = get_feature_groups(train_df, target=target)
    preprocessor = build_preprocessor(numerical_features, categorical_features)

    X, y = split_features_target(train_df, target=target)
    X_train, X_val, y_train, y_val = make_train_validation_split(X, y)

    tuning_results = []

    for alpha in alpha_values:
        model = Pipeline(steps=[
            ("preprocessor", preprocessor),
            ("model", Ridge(alpha=alpha)),
        ])
        metrics = evaluate_regression_model(model, X_train, y_train, X_val, y_val)
        tuning_results.append({"alpha": alpha, **metrics})

    return pd.DataFrame(tuning_results).sort_values("RMSE")

# ridge_tuning_results = tune_ridge_alpha(train_df)
# ridge_tuning_results

## Original Assessment Run Summary

In the original run, Ridge Regression achieved the best overall validation performance after tuning.

| Model | Validation RMSE | Validation MAE | Validation R² |
|---|---:|---:|---:|
| Linear Regression | 103.684 | 47.816 | 0.3955 |
| Ridge Regression, alpha = 1 | 103.669 | 47.729 | 0.3957 |
| Lasso Regression, alpha = 1 | 103.983 | 46.801 | 0.3920 |

The tuned Ridge model with `alpha = 10` achieved:

| Tuned Model | Validation RMSE | Validation MAE | Validation R² |
|---|---:|---:|---:|
| Ridge Regression, alpha = 10 | 103.616 | 47.429 | 0.3963 |

The improvement over ordinary Linear Regression was small, but Ridge Regression gave the best balance between validation error and generalisation.

## Final Model Template

The function below trains the selected final model. It is included as a reusable template, but it will only run when compatible local data is available.

In [ ]:
def train_final_model(train_df, alpha=10, target=TARGET):
    numerical_features, categorical_features = get_feature_groups(train_df, target=target)
    preprocessor = build_preprocessor(numerical_features, categorical_features)

    X, y = split_features_target(train_df, target=target)

    final_model = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", Ridge(alpha=alpha)),
    ])

    final_model.fit(X, y)
    return final_model

# final_model = train_final_model(train_df, alpha=10)

## Real-World Suitability and Limitations

This model can be useful as a basic decision-support tool for estimating Airbnb prices. However, it should not be used as a fully automated pricing system.

Main limitations:

- Linear models may struggle with expensive or unusual listings.
- The target distribution is right-skewed.
- Price may depend on unobserved factors such as property quality, events, seasonality, photos, and host strategy.
- Historical pricing patterns may reflect social and economic inequalities.

Responsible use requires transparency. A predicted price should be treated as an estimate that supports human judgement, not as a guaranteed correct value.

## Conclusion

This project demonstrates an end-to-end regression workflow for Airbnb price prediction. The workflow includes data inspection, exploratory analysis, preprocessing, model comparison, Ridge hyperparameter tuning, and evaluation using RMSE, MAE, and R².

Ridge Regression with `alpha = 10` was selected as the final model in the original analysis because it achieved the best validation RMSE and R² among the tested models.